# Calibration Module — Workflow Demo

This notebook demonstrates the conformal-prediction threshold calibration pipeline for CH4Net v8.
Two classes are shown:

1. **`ConformalCalibrator`** — Global split conformal prediction (Angelopoulos & Bates 2021).  
   Computes τ such that FPR ≤ α with finite-sample guarantees, adds bootstrap CI, and
   a sample-size adequacy check.

2. **`MondrianConformalCalibrator`** — Stratified (Mondrian) extension.  
   Repeats calibration independently per ecoregion **and** per CLC land-cover class,
   issuing `[SMALL_N]` warnings where n < 5.

**Prerequisites:** no third-party libraries required (matplotlib optional for plots).  
Run from the `methane-api/` repo root **or** from `scripts/calibration/`.

In [ ]:
import sys
from pathlib import Path

# Resolve repo root regardless of working directory
repo_root = Path.cwd()
while repo_root.name not in ('methane-api', '') and repo_root != repo_root.parent:
    repo_root = repo_root.parent
scripts_dir = repo_root / 'scripts'
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

print(f'Repo root resolved to: {repo_root}')

In [ ]:
from calibration.conformal_threshold import (
    NonEmitterScoreLoader,
    ConformalCalibrator,
    MondrianConformalCalibrator,
    make_calibration_plot,
    SCORES_PATH,
    ALPHA_LEVELS,
    PRODUCTION_THRESHOLD,
)

print('Imports OK')
print(f'Default scores path : {SCORES_PATH}')
print(f'Alpha levels        : {ALPHA_LEVELS}')
print(f'Production threshold: {PRODUCTION_THRESHOLD}')

## Part 1 — Load and inspect the calibration scores

`NonEmitterScoreLoader` reads `results_analysis/nonemitter_sc_scores.json`,
filters to records with `status='ok'`, and exposes a clean score list.

In [ ]:
loader = NonEmitterScoreLoader(SCORES_PATH).Load()

desc = loader.Describe()
print(f"Valid records   : {desc['n']}")
print(f"Score range     : [{desc['min']:.4f}, {desc['max']:.4f}]")
print(f"Score mean/med  : {desc['mean']:.4f} / {desc['median']:.4f}")
print(f"Score std       : {desc['std']:.4f}")

## Part 2 — Global ConformalCalibrator

### 2a. Fit at primary alpha = 0.10

In [ ]:
calibrator = ConformalCalibrator()
results = calibrator.Run(loader, alpha=0.10)

calibrator.PrintSummary(results)

### 2b. Inspect all alpha levels

In [ ]:
import math

scores = loader.scores
n = len(scores)

print(f"{'alpha':>8}  {'rank':>6}  {'tau':>10}  {'FPR @ prod_tau':>16}")
print('-' * 50)
for alpha in ALPHA_LEVELS:
    rank = math.ceil((n + 1) * (1 - alpha))
    tau = calibrator.Fit(scores, alpha)
    fpr = calibrator.EmpiricalFpr(scores, PRODUCTION_THRESHOLD)
    print(f"{alpha:>8.2f}  {rank:>6d}  {tau:>10.4f}  {fpr:>16.3f}")

### 2c. Bootstrap confidence interval on τ (alpha = 0.10)

In [ ]:
boot = calibrator.Bootstrap(scores, alpha=0.10, n_boot=2000, seed=42)

print(f"Bootstrap n_resamples : {boot['n_bootstrap']}")
print(f"Bootstrap mean tau    : {boot['tau_boot_mean']:.4f}")
print(f"Bootstrap std tau     : {boot['tau_boot_std']:.4f}")
print(f"90% CI                : [{boot['tau_boot_p5']:.4f}, {boot['tau_boot_p95']:.4f}]")

### 2d. Sample size adequacy check

In [ ]:
ssa = calibrator.SampleSizeAnalysis(n=n, alpha=0.10)

print(f"n (current)   : {ssa['n_calibration']}")
print(f"n (minimum)   : {ssa['min_n_nontrivial']}   (for non-trivial bound at alpha=0.10)")
print(f"Adequate?     : {ssa['n_adequate']}")

## Part 3 — MondrianConformalCalibrator (stratified)

Runs the global calibrator first, then fits independently per ecoregion and per CLC land-cover class.
Groups with n < 5 receive a `[SMALL_N]` warning in the output.

In [ ]:
mondrian = MondrianConformalCalibrator()
m_results = mondrian.Run(loader, alpha=0.10)

mondrian.PrintSummary(m_results)

### 3b. Inspect Mondrian tau values directly

In [ ]:
print('Ecoregion thresholds:')
for group, data in m_results['mondrian_by_ecoregion'].items():
    flag = '  [SMALL_N]' if data['small_n_warning'] else ''
    print(f"  {group:<25} n={data['n']:>2}  tau={data['tau']:.4f}{flag}")

print('\nCLC land-cover thresholds:')
for group, data in m_results['mondrian_by_clc_class'].items():
    flag = '  [SMALL_N]' if data['small_n_warning'] else ''
    print(f"  {group:<25} n={data['n']:>2}  tau={data['tau']:.4f}{flag}")

## Part 4 — ECDF calibration plot

In [ ]:
    plot_path = repo_root / 'results_analysis' / 'conformal_calibration_demo.png'
    # make_calibration_plot expects {float_alpha: float_tau}; transform from the
    # nested JSON structure {"tau_alpha_10": {"alpha": 0.10, "tau": 3.5796, ...}, ...}
    thresholds = {
        info["alpha"]: info["tau"]
        for info in m_results["global_thresholds"].values()
    }
    make_calibration_plot(
        scores=loader.scores,
        global_thresholds=thresholds,
        records=loader.records,
        out_path=plot_path,
    )

## Part 5 — Save results

In [ ]:
mondrian.SaveResults(m_results)
print('Results saved to results_analysis/calibrated_threshold.json')